# Exploring the DataCollector Class

This notebook explores the functionality of the `DataCollector` class, which automatically collects geospatial data from various Web Feature Service (WFS) sources.

## Overview

The `DataCollector` class:
- Connects to multiple WFS services (land use, buildings, vegetation)
- Fetches geospatial data within a specified area of interest
- Handles pagination automatically for large datasets
- Returns organized GeoDataFrames ready for analysis

**Note:** DataCollector is currently used in `src/data/data_handler.py` but not in any notebooks yet.


In [1]:
import sys
sys.path.append("../")

import geopandas as gpd
import folium
from shapely.geometry import Polygon, Point
import pandas as pd

import src.constants as CONST
import src.config as CONFIG
import src.data.data_collector as DC
import src.data.schema_wfs_service as SWS


## 1. Check WFS Services Configuration

First, let's examine what WFS services are configured and what layers they provide.


In [ ]:
print(f"Total number of WFS services configured: {len(CONFIG.KNOWN_WFS_SERVICES)}")
print("\n" + "="*80)

for i, service in enumerate(CONFIG.KNOWN_WFS_SERVICES, 1):
    print(f"\n{i}. Service Name: {service.name}")
    print(f"   URL: {service.url}")
    print(f"   Version: {service.version}")
    print(f"   Relevant Layers ({len(service.relevant_layers)}):")
    for layer in service.relevant_layers:
        print(f"      - {layer}")


Total number of WFS services configured: 3


1. Service Name: land_use
   URL: https://service.pdok.nl/rvo/brpgewaspercelen/wfs/v1_0
   Version: 1.0.0
   Relevant Layers (1):
      - BrpGewas

2. Service Name: building_location
   URL: https://service.pdok.nl/lv/bag/wfs/v2_0
   Version: 2.0.0
   Relevant Layers (1):
      - bag:pand

3. Service Name: vegetation
   URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_vegetatielegger/ows?version=2.0.0
   Version: 2.0.0
   Relevant Layers (3):
      - rws_vegetatielegger:bomen
      - rws_vegetatielegger:heggen
      - rws_vegetatielegger:vegetatieklassen


## 2. Define Test Area of Interest

We'll use an area near Zaltbommel, Netherlands (along the Waal river), which is used in the test fixtures. This is a relevant area for river bank erosion studies.

**About the buffer:** The `buffer_in_metres` parameter expands the source shape to capture surrounding context. For example, if studying a river bank point, you might want data from 100-1000m around it to understand nearby buildings, vegetation, and land use.


In [ ]:
# Create a test polygon near Zaltbommel (coordinates from test fixtures)
# This is a small area along the Waal river
test_area_wgs84 = Polygon([
    (5.4639, 51.8875),  # Southwest corner
    (5.4664, 51.8879),  # Northwest corner
    (5.4665, 51.8872),  # Northeast corner
    (5.4634, 51.8869),  # Southeast corner
    (5.4639, 51.8875)   # Close the polygon
])

print(f"Test area (WGS84): {test_area_wgs84}")
print(f"Area (approx): {test_area_wgs84.area * 111000 * 111000:.0f} m²")  # Rough conversion


Test area (WGS84): POLYGON ((5.4639 51.8875, 5.4664 51.8879, 5.4665 51.8872, 5.4634 51.8869, 5.4639 51.8875))
Area (approx): 21562 m²


In [ ]:
# Visualize the test area on a map
test_gdf = gpd.GeoDataFrame([1], geometry=[test_area_wgs84], crs=CONST.EPSG_WGS84)

m = folium.Map(
    location=[51.887, 5.465],  # Center of our test area
    zoom_start=16,
    control_scale=True
)

folium.GeoJson(test_gdf).add_to(m)
folium.Marker(
    [51.887, 5.465],
    popup="Test Area - Near Zaltbommel, Waal River"
).add_to(m)

m


## 3. Initialize DataCollector

Now we'll create a DataCollector instance. We'll use a buffer of 500 meters to capture surrounding context.


In [ ]:
# Initialize the DataCollector
buffer_size = 500  # meters - expands the area to get surrounding context

data_collector = DC.DataCollector(
    source_shape=test_area_wgs84,
    source_epsg_crs=CONST.EPSG_WGS84,
    buffer_in_metres=buffer_size,
    wfs_services=CONFIG.KNOWN_WFS_SERVICES,
)

print("DataCollector initialized successfully!")
print(f"\nSource shape (original): {data_collector.source_shape_raw}")
print(f"Source shape (with buffer): {data_collector.source_shape}")
print(f"Buffer size: {data_collector.buffer} meters")
print(f"\nNumber of WFS services initialized: {len(data_collector.wfs_services)}")
print(f"WFS service names: {list(data_collector.wfs_services.keys())}")


DataCollector initialized successfully!

Source shape (original): POLYGON ((5.4639 51.8875, 5.4664 51.8879, 5.4665 51.8872, 5.4634 51.8869, 5.4639 51.8875))
Source shape (with buffer): POLYGON ((5.457444648045475 51.88955957887116, 5.457783650979745 51.88992358918429, 5.458176113076415 51.890266420114095, 5.458618605212322 51.89058507516054, 5.459107260732722 51.89087676907114, 5.459637809232509 51.89113895219983, 5.46020561387734 51.89136933280736, 5.460805711938405 51.8915658971071, 5.461432858186156 51.891726926880914, 5.462081570762945 51.89185101451052, 5.464581796501881 51.89225105293777, 5.465267273005334 51.89233915643703, 5.4659632761540955 51.89238601428322, 5.4666633378805525 51.89239119101851, 5.467360952366372 51.89235463853495, 5.468049636556374 51.892276696521755, 5.468722990459927 51.892158089305845, 5.469374756676959 51.89199991911462, 5.469998878593151 51.89180365582404, 5.470589556701273 51.89157112328788, 5.47114130252356 51.89130448237538, 5.471648989632511 51.8910

In [ ]:
# Verify we can connect to the WFS services and see available layers
print("Verifying WFS service connections...")
print("="*80)

for service_name, wfs_service in data_collector.wfs_services.items():
    print(f"\n{service_name}:")
    try:
        available_layers = list(wfs_service.contents.keys())
        print(f"  ✓ Connected successfully")
        print(f"  Available layers: {len(available_layers)}")
        
        # Find the configured relevant layers
        for raw_service in data_collector.wfs_services_raw:
            if raw_service.name == service_name:
                relevant_layers = raw_service.relevant_layers
                print(f"  Configured relevant layers: {relevant_layers}")
                
                # Check if all configured layers exist
                missing = set(relevant_layers) - set(available_layers)
                if missing:
                    print(f"  ⚠ Warning: Missing layers: {missing}")
                else:
                    print(f"  ✓ All configured layers are available")
                break
    except Exception as e:
        print(f"  ✗ Connection failed: {e}")


Verifying WFS service connections...

land_use:
  ✓ Connected successfully
  Available layers: 1
  Configured relevant layers: ['BrpGewas']
  ✓ All configured layers are available

building_location:
  ✓ Connected successfully
  Available layers: 5
  Configured relevant layers: ['bag:pand']
  ✓ All configured layers are available

vegetation:
  ✓ Connected successfully
  Available layers: 3
  Configured relevant layers: ['rws_vegetatielegger:bomen', 'rws_vegetatielegger:heggen', 'rws_vegetatielegger:vegetatieklassen']
  ✓ All configured layers are available


## 4. Collect Data from All WFS Services

Now we'll fetch data from all configured WFS services. This may take a while depending on:
- The size of the area
- The number of features in that area
- Network speed
- WFS service response times

The DataCollector automatically handles pagination if services limit the number of features per request.


In [ ]:
print("Starting data collection from all WFS services...")
print("This may take a few minutes depending on the area size and data availability.\n")

# Check initial state
print(f"Initial relevant_geospatial_data: {len(data_collector.relevant_geospatial_data)} services")

# Collect data from all WFS services
data_collector.get_data_from_all_wfs()

print("\n" + "="*80)
print("Data collection completed!")
print(f"Number of services with data: {len(data_collector.relevant_geospatial_data)}")


Starting data collection from all WFS services...
This may take a few minutes depending on the area size and data availability.

Initial relevant_geospatial_data: 0 services

Data collection completed!
Number of services with data: 3


## 5. Explore the Collected Data Structure

The `relevant_geospatial_data` is a nested dictionary:
- First level: WFS service names (e.g., 'land_use', 'building_location', 'vegetation')
- Second level: Layer names (e.g., 'BrpGewas', 'bag:pand', 'rws_vegetatielegger:bomen')
- Values: GeoDataFrames with the actual geospatial data


In [10]:
print("Data Structure Overview:")
print("="*80)

for wfs_service_name in data_collector.relevant_geospatial_data:
    print(f"\n📦 {wfs_service_name}")
    service_data = data_collector.relevant_geospatial_data[wfs_service_name]
    
    for layer_name, gdf in service_data.items():
        print(f"  └─ {layer_name}")
        print(f"     Features: {len(gdf)}")
        try:
            crs_str = gdf.crs
        except:
            crs_str = 'N/A'
        print(f"     CRS: {crs_str}")
        if len(gdf) > 0:
            print(f"     Columns: {list(gdf.columns)}")
            print(f"     Geometry type: {gdf.geometry.type.iloc[0] if len(gdf) > 0 else 'N/A'}")
        else:
            print(f"     ⚠ Empty GeoDataFrame (no features found in this area)")


Data Structure Overview:

📦 land_use
  └─ BrpGewas
     Features: 41
     CRS: EPSG:28992
     Columns: ['geometry', 'category', 'gewas', 'gewascode', 'jaar', 'status']
     Geometry type: Polygon

📦 building_location
  └─ bag:pand
     Features: 3
     CRS: EPSG:28992
     Columns: ['geometry', 'identificatie', 'rdf_seealso', 'bouwjaar', 'status', 'gebruiksdoel', 'aantal_verblijfsobjecten', 'oppervlakte_min', 'oppervlakte_max']
     Geometry type: Polygon

📦 vegetation
  └─ rws_vegetatielegger:bomen
     Features: 16
     CRS: EPSG:28992
     Columns: ['geometry', 'objectid', 'fi_code_p', 'gdb_geomattr_data']
     Geometry type: Point
  └─ rws_vegetatielegger:heggen
     Features: 0
     CRS: N/A
     ⚠ Empty GeoDataFrame (no features found in this area)
  └─ rws_vegetatielegger:vegetatieklassen
     Features: 86
     CRS: EPSG:28992
     Columns: ['geometry', 'vlklasse', 'gdb_geomattr_data']
     Geometry type: Polygon


## 6. Sample Data from Each Layer

Let's look at sample data from each layer to understand what information is available.


In [22]:
for wfs_service_name in data_collector.relevant_geospatial_data:
    service_data = data_collector.relevant_geospatial_data[wfs_service_name]

    for layer_name, gdf in service_data.items():
        print(layer_name)
        print(type(gdf))
        print(f"\n{'='*80}")
        print(f"{wfs_service_name} → {layer_name}")
        print(f"{'='*80}")
        
        if len(gdf) == 0:
            print("No data available for this layer in the specified area.")
            continue
        
        print(f"\nFirst few rows:")
        display(gdf.head())
        
        print(f"\nDataFrame info:")
        gdf.info()


BrpGewas
<class 'geopandas.geodataframe.GeoDataFrame'>

land_use → BrpGewas

First few rows:


,geometry,category,gewas,gewascode,jaar,status
0,"POLYGON ((160439.779 432942.67, 160440.066 432...",Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief
1,"POLYGON ((160191.187 432575.748, 160193.006 43...",Landschapselement,Sloot,343,2024,Definitief
2,"POLYGON ((160440.006 432942.772, 160479.179 43...",Grasland,"Grasland, blijvend",265,2024,Definitief
3,"POLYGON ((160777.153 432641.746, 160780.6 4326...",Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief
4,"POLYGON ((160367.475 432658.533, 160376.54 432...",Bouwland,"Mais, snij-",259,2024,Definitief



DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   geometry   41 non-null     geometry
 1   category   41 non-null     object  
 2   gewas      41 non-null     object  
 3   gewascode  41 non-null     int64   
 4   jaar       41 non-null     int64   
 5   status     41 non-null     object  
dtypes: geometry(1), int64(2), object(3)
memory usage: 2.1+ KB
bag:pand
<class 'geopandas.geodataframe.GeoDataFrame'>

building_location → bag:pand

First few rows:


,geometry,identificatie,rdf_seealso,bouwjaar,status,gebruiksdoel,aantal_verblijfsobjecten,oppervlakte_min,oppervlakte_max
0,"POLYGON ((160717.001 432682.78, 160714.519 432...",0668100000010867,http://bag.basisregistraties.overheid.nl/bag/i...,2004,Pand in gebruik,,0,NaN,NaN
1,"POLYGON ((160476.018 433775.868, 160478.007 43...",1740100000000018,http://bag.basisregistraties.overheid.nl/bag/i...,1955,Pand in gebruik,woonfunctie,1,585.0,585.0
2,"POLYGON ((159814.847 433509.007, 159814.024 43...",0281100000025121,http://bag.basisregistraties.overheid.nl/bag/i...,1982,Pand in gebruik,industriefunctie,1,428.0,428.0



DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   geometry                  3 non-null      geometry
 1   identificatie             3 non-null      object  
 2   rdf_seealso               3 non-null      object  
 3   bouwjaar                  3 non-null      int64   
 4   status                    3 non-null      object  
 5   gebruiksdoel              3 non-null      object  
 6   aantal_verblijfsobjecten  3 non-null      int64   
 7   oppervlakte_min           2 non-null      float64 
 8   oppervlakte_max           2 non-null      float64 
dtypes: float64(2), geometry(1), int64(2), object(4)
memory usage: 348.0+ bytes
rws_vegetatielegger:bomen
<class 'geopandas.geodataframe.GeoDataFrame'>

vegetation → rws_vegetatielegger:bomen

First few rows:


,geometry,objectid,fi_code_p,gdb_geomattr_data
0,POINT (160412.82 432663.564),54334,Boom5+,None
1,POINT (160425.381 432670.97),54335,Boom5+,None
2,POINT (160771.595 432725.83),54351,Boom5+,None
3,POINT (159912.466 432750.435),54361,Boom5+,None
4,POINT (160957.648 432791.773),54390,Boom5+,None



DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   geometry           16 non-null     geometry
 1   objectid           16 non-null     int64   
 2   fi_code_p          16 non-null     object  
 3   gdb_geomattr_data  0 non-null      object  
dtypes: geometry(1), int64(1), object(2)
memory usage: 644.0+ bytes
rws_vegetatielegger:heggen
<class 'geopandas.geodataframe.GeoDataFrame'>

vegetation → rws_vegetatielegger:heggen
No data available for this layer in the specified area.
rws_vegetatielegger:vegetatieklassen
<class 'geopandas.geodataframe.GeoDataFrame'>

vegetation → rws_vegetatielegger:vegetatieklassen

First few rows:


,geometry,vlklasse,gdb_geomattr_data
0,"POLYGON Z ((160670.478 432781.543 0, 160665.11...",Bos,[B@2d96942f
1,"POLYGON Z ((160856.749 432669.534 0, 160856.99...",Bos,[B@47776dee
2,"POLYGON Z ((160958.775 432677.003 0, 160959.85...",Bos,[B@3985f81e
3,"POLYGON Z ((160573.923 432753.533 0, 160581.66...",Bos,[B@3b454bab
4,"POLYGON Z ((160632.118 432758.181 0, 160631.98...",Bos,[B@22eb8700



DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   geometry           86 non-null     geometry
 1   vlklasse           86 non-null     object  
 2   gdb_geomattr_data  86 non-null     object  
dtypes: geometry(1), object(2)
memory usage: 2.1+ KB


## 7. Visualize Collected Data

Create maps showing the collected data overlaid on the test area.


In [29]:
# Create a map with all layers
m = folium.Map(
    location=[51.887, 5.465],
    zoom_start=15,
    control_scale=True
)

# Add the test area
test_gdf_wgs84 = gpd.GeoDataFrame([1], geometry=[test_area_wgs84], crs=CONST.EPSG_WGS84)
folium.GeoJson(
    test_gdf_wgs84,
    style_function=lambda feature: {
        'fillColor': 'yellow',
        'color': 'black',
        'weight': 3,
        'fillOpacity': 0.3
    }
).add_to(m)

# Add each layer as a feature group
colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown']
color_idx = 0

for wfs_service_name in data_collector.relevant_geospatial_data:
    service_data = data_collector.relevant_geospatial_data[wfs_service_name]
    
    for layer_name, gdf in service_data.items():
        if len(gdf) == 0:
            continue
        
        # Convert to WGS84 for folium
        gdf_wgs84 = gdf.to_crs(epsg=CONST.EPSG_WGS84)
        
        # Create feature group
        fg = folium.FeatureGroup(
            name=f"{wfs_service_name}: {layer_name}",
            show=True
        ).add_to(m)
        
        # Get color for this layer (capture in closure)
        current_color = colors[color_idx % len(colors)]
        
        # Add geometry with proper color closure
        def make_style(color):
            return lambda feature: {
                'fillColor': color,
                'color': 'black',
                'weight': 1,
                'fillOpacity': 0.5
            }
        
        folium.GeoJson(
            gdf_wgs84,
            style_function=make_style(current_color)
        ).add_to(fg)
        
        color_idx += 1

# Add layer control
folium.LayerControl().add_to(m)

m


## 8. Summary Statistics

Let's create a summary of what was collected.


In [ ]:
summary_data = []

for wfs_service_name in data_collector.relevant_geospatial_data:
    service_data = data_collector.relevant_geospatial_data[wfs_service_name]
    
    for layer_name, gdf in service_data.items():
        summary_data.append({
            'WFS Service': wfs_service_name,
            'Layer': layer_name,
            'Feature Count': len(gdf),
            'CRS': str(gdf.crs) if gdf.crs else 'None',
            'Geometry Type': gdf.geometry.type.iloc[0] if len(gdf) > 0 else 'N/A',
            'Has Data': len(gdf) > 0
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"\nTotal features collected: {summary_df['Feature Count'].sum()}")
print(f"Layers with data: {summary_df['Has Data'].sum()} / {len(summary_df)}")


## 9. Test Individual Service/Layer Collection (Optional)

You can also test collecting data from a single service or layer. This is useful for debugging or when you only need specific data.


In [ ]:
# Example: Get data from just one WFS service
if len(CONFIG.KNOWN_WFS_SERVICES) > 0:
    test_service_name = CONFIG.KNOWN_WFS_SERVICES[0].name
    print(f"Testing collection from single service: {test_service_name}")
    
    # Create a new collector for this test
    test_collector = DC.DataCollector(
        source_shape=test_area_wgs84,
        source_epsg_crs=CONST.EPSG_WGS84,
        buffer_in_metres=500,
        wfs_services=[CONFIG.KNOWN_WFS_SERVICES[0]],  # Just one service
    )
    
    # Get data from that single service
    single_service_data = test_collector.load_data_from_single_wfs(test_service_name)
    
    print(f"\nCollected {len(single_service_data)} layers from {test_service_name}")
    for layer_name, gdf in single_service_data.items():
        print(f"  - {layer_name}: {len(gdf)} features")
